# 文書を検索して回答を生成する

先に [PDF を Chroma に登録する](8300.1101.RAG.build_chroma_db.ipynb)を実行します。`OPENAI_API_KEY` を設定した環境で Jupyter を起動し、質問ファイルを確認します。

In [1]:
import json
import os
from pathlib import Path
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

## 処理を関数として定義する

In [2]:
def load_questions(path: str) -> list[str]:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Question file not found: {path}")

    questions = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            q = line.strip()
            if q and not q.startswith("#"):
                questions.append(q)

    return questions


def make_context(docs) -> str:
    blocks = []

    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "unknown")

        blocks.append(
            f"[Document {i}]\n"
            f"source: {source}\n"
            f"page: {page}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(blocks)

In [3]:
def answer_questions(
    db_dir: str,
    question_file: str,
    output_file: str | None = None,
    collection_name: str = "pdf_rag",
    embedding_model: str = "text-embedding-3-small",
    llm_model: str = "gpt-5.4-mini",
    k: int = 5,
) -> None:
    db_dir = Path(db_dir)

    if not db_dir.exists():
        raise FileNotFoundError(f"Chroma DB directory not found: {db_dir}")

    if os.getenv("OPENAI_API_KEY") is None:
        raise RuntimeError("OPENAI_API_KEY is not set")

    questions = load_questions(question_file)

    embeddings = OpenAIEmbeddings(model=embedding_model)

    vectorstore = Chroma(
        persist_directory=str(db_dir),
        embedding_function=embeddings,
        collection_name=collection_name,
    )

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": k}
    )

    llm = ChatOpenAI(
        model=llm_model,
        temperature=0,
    )

    results = []

    for i, question in enumerate(questions, start=1):
        docs = retriever.invoke(question)
        context = make_context(docs)

        prompt = f"""
1.まず、以下のcontextだけを根拠に質問に日本語で答えてください。
2.もし、1で回答できなかった場合は、「文書中には見つかりませんが、contextから推測します。」と書いてから、contextを元に推測してください。

<context>
{context}
</context>

質問:
{question}
"""

        response = llm.invoke(prompt)
        answer = response.content

        print("=" * 80)
        print(f"Q{i}: {question}")
        print()
        print(answer)
        print()

        sources = []
        for doc in docs:
            sources.append(
                {
                    "source": doc.metadata.get("source"),
                    "page": doc.metadata.get("page"),
                    "content_preview": doc.page_content[:300],
                }
            )

        results.append(
            {
                "question": question,
                "answer": answer,
                "sources": sources,
            }
        )

    if output_file is not None:
        output_path = Path(output_file)
        output_path.write_text(
            json.dumps(results, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        print(f"Saved results: {output_path}")

## 実習の入力と設定を確認する

In [4]:
question_file = Path("examples/ARiSE_SPReAD/questions.txt")
db_dir = Path("chroma_db")
output_file = Path("answers.json")
collection_name = "pdf_rag"
embedding_model = "text-embedding-3-small"
llm_model = "gpt-4.1-mini"
k = 5

for question in load_questions(question_file):
    print(question)

ARiSE申請の重複制限はありますか？詳細を教えてください。
SPReAD申請の重複制限はありますか？詳細を教えてください。
ARiSEの研究開発分担者はSPReADの研究代表者になれますか？


## 関連箇所を検索して回答する

各質問につき Chroma から関連するチャンクを最大 `k` 件取得し、LLM に渡します。回答と出典情報を `answers.json` に保存します。この処理では OpenAI の埋め込みと文章生成の API を使います。

In [5]:
if not os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY を設定してから、このセルを実行してください。")
elif not db_dir.is_dir():
    print("先に PDF 登録ノートブックで Chroma データベースを作成してください。")
else:
    answer_questions(
        db_dir=db_dir,
        question_file=question_file,
        output_file=output_file,
        collection_name=collection_name,
        embedding_model=embedding_model,
        llm_model=llm_model,
        k=k,
    )
    results = json.loads(output_file.read_text(encoding="utf-8"))
    for result in results:
        print(result["question"])
        for source in result["sources"]:
            print(f'  {source["source"]} (ページ番号: {source["page"]}, 0 始まり)')

Q1: ARiSE申請の重複制限はありますか？詳細を教えてください。

ARiSE申請の重複制限について、以下の内容がcontextから確認できます。

1. 研究開発代表者として応募できるのは本事業全体で1件のみです。（Document 1, 2）
2. AI for Science 萌芽的挑戦研究創出事業（SPReAD）の研究代表者が同時にARiSEの共同代表者を務めることはできません。（Document 1, 2）
3. 研究開発代表者と共同研究開発代表者が互いに入れ替わって複数件の応募をすることはできません。（Document 1, 2）
4. 研究開発代表者または共同研究開発代表者として応募し、かつ他の提案において共同研究開発代表者または研究開発分担者として応募し、両方の研究開発課題が採択された場合は、研究費の減額や参画課題の調整が行われることがあります。（Document 1, 2）
5. その他の競争的研究費制度との間に重複応募の制限はありませんが、SPReADとARiSE間での重複制限はあります。（Document 1, 2, 3）

まとめると、ARiSE申請においては、研究開発代表者としては本事業全体で1件のみ応募可能であり、SPReADの研究代表者が同時にARiSEの共同代表者を務めることはできません。また、代表者や共同代表者の役割を入れ替えて複数応募することも禁止されています。複数の役割で採択された場合は調整が行われる可能性があります。

以上がARiSE申請に関する重複制限の詳細です。

Q2: SPReAD申請の重複制限はありますか？詳細を教えてください。

SPReAD申請の重複制限について、以下の通りです。

1. 研究開発代表者として応募できるのは、本事業全体で1件のみです。  
2. AI for Science 萌芽的挑戦研究創出事業（SPReAD）の研究代表者が同時にARiSEの共同代表者を務めることはできません。  
3. その他の競争的研究費制度との間に重複応募の制限はありません。  
4. 共同研究開発代表者または研究開発分担者として参画する場合には以下の制限があります。  
　a. 研究開発代表者と共同研究開発代表者が互いに入れ替わって複数件の応募をすることはできません。  
　b. 研究開発代表者または共同研究開発代